In [17]:
# %% [markdown]
# CDPKit Conformer Generation & Rosetta Ranking Pipeline
# 
# This notebook demonstrates:
# 1. Pulling .params files from the Enamine REAL library
# 2. Generating 150 conformers per ligand using CDPKit
# 3. Ranking each conformer with Rosetta energy scores

# %%
from pathlib import Path
import sys, os

# Point to the discovery module
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'func', 'discovery'))
from cdpkit_conformer_pipeline import (
    extract_params_from_enamine,
    generate_cdpkit_conformers,
    convert_conformer_to_params,
    score_single_conformer_with_rosetta,
)
# ## Configuration
# Adjust these paths for your setup:

# ── Input ──────────────────────────────────────────────────────────
INPUT_LIST    = "conformer_input.txt"          # shapedb results list
TARGET_PDB    = Path.absolute(Path("../input_pdb/processed/orexin_lemborexant_receptor_R.pdb"))  # change to your target
ANCHOR_RESIDUES = "100"                           # anchor residue(s) need generic number after renumbering
MOTIFS_FILE   = Path.absolute(Path("../motifs/FINAL_motifs_list_filtered_2_3_2023.motifs"))
EXTRA_ARG = Path.absolute(Path("../extra_arg/sample_extra_arg.txt"))  # example extra arg file for testing

# ── Paths ──────────────────────────────────────────────────────────
REALM_LOCATION = "/pi/summer.thyme-umw/Ji_rosetta_discovery"
ENAMINE_PATH   = "/pi/summer.thyme-umw/enamine-REAL-2.6billion"
OUTPUT_DIR     = Path.absolute(Path("./output/cdpkit_test"))

# ── Parameters ─────────────────────────────────────────────────────
NUM_CONFORMERS = 150
TOP_N          = 20
ATR, REP, DDG  = -2.0, 150.0, -9.0

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

Output directory: /pi/summer.thyme-umw/Ji_rosetta_discovery/test/output/cdpkit_test


In [2]:
## Step 1: Read input list
entries = []
with open(INPUT_LIST, 'r') as fh:
    for line in fh:
        print(line.rstrip())
        line = line.strip()
        if not line:
            continue
        fields = [f.strip() for f in line.split(',')]
        if len(fields) < 4:
            continue
        score, ligand_conf, chunk, subchunk = fields[:4]
        ligand_name = "_".join(ligand_conf.split('_')[:-1])
        try:
            conf_num = int(ligand_conf.split('_')[-1])
            entries.append((float(score), ligand_name, conf_num, chunk, subchunk))
        except ValueError:
            continue

for s, name, cn, ch, sc in entries:
    print(f"  {name}  conf={cn}  chunk={ch}  subchunk={sc}  score={s:.4f}")


-0.433132231236,PV-006439055682_1,33229,0
  PV-006439055682  conf=1  chunk=33229  subchunk=0  score=-0.4331


In [3]:

# ## Step 2: Extract params from Enamine library

# %%
params_dir = os.path.join(OUTPUT_DIR, 'extracted_params')
os.makedirs(params_dir, exist_ok=True)

results = []
for score, ligand_name, conf_num, chunk, subchunk in entries:
    print(f"\n--- {ligand_name} (conf {conf_num}) ---")
    pf = extract_params_from_enamine(
        ligand_name, conf_num, chunk, subchunk,
        ENAMINE_PATH, params_dir
    )
    if pf:
        results.append((score, ligand_name, conf_num, chunk, subchunk, pf))
        print(f"  ✓ Extracted: {os.path.basename(pf)}")
    else:
        print(f"  ✗ Failed to extract params")

print(f"\nSuccessfully extracted {len(results)}/{len(entries)} params files")




--- PV-006439055682 (conf 1) ---
  ✓ Extracted: PV-006439055682_1.params

Successfully extracted 1/1 params files


In [4]:
## Step 3: Extract SMILES from params

from rdkit import Chem

# %%
from cdpkit_conformer_pipeline  import (
    extract_smiles_from_params
)

smiles_map = {}
for score, ligand_name, conf_num, chunk, subchunk, pf in results:
    with open(pf, 'r') as fh:
        params_text = fh.read()
    
    smiles = extract_smiles_from_params(params_text)
    if smiles:
        smiles_map[ligand_name] = smiles
        print(f"  {ligand_name}: {smiles}")
    else:
        print(f"  {ligand_name}: ✗ Could not extract SMILES")


    Parsed 47 atoms, 50 bonds from params
    Generated SMILES via RDKit: [H]OC([H])([H])C([H])(C([H])([H])c1c([H])nc([H])c([H])c1[H])C([H])([H])N([H])C(=O)c1c([H])c([H])c([H])c2c1oc1c([H])c([H])c([H])c([H])c12
  PV-006439055682: [H]OC([H])([H])C([H])(C([H])([H])c1c([H])nc([H])c([H])c1[H])C([H])([H])N([H])C(=O)c1c([H])c([H])c([H])c2c1oc1c([H])c([H])c([H])c([H])c12


In [5]:
# %% [markdown]
# ## Step 4: Generate conformers with CDPKit
from cdpkit_conformer_pipeline import generate_cdpkit_conformers

all_conformers = {}
for ligand_name, smiles in smiles_map.items():
    print(f"\nGenerating {NUM_CONFORMERS} conformers for {ligand_name}...")
    conformers = generate_cdpkit_conformers(smiles, NUM_CONFORMERS)
    if conformers:
        all_conformers[ligand_name] = conformers
        print(f"  ✓ Generated {len(conformers)} conformers")
    else:
        print(f"  ✗ Failed to generate conformers")

print(f"\nGenerated conformers for {len(all_conformers)}/{len(smiles_map)} ligands")


Generating 150 conformers for PV-006439055682...
  CDPKit generated 150 conformers from SMILES
  ✓ Generated 150 conformers

Generated conformers for 1/1 ligands


In [6]:

# ## Step 5: Convert conformers to Rosetta .params
from cdpkit_conformer_pipeline import convert_conformer_to_params

conf_params_dir = os.path.join(OUTPUT_DIR, 'conformer_params')
os.makedirs(conf_params_dir, exist_ok=True)

all_params = {}  # ligand_name -> [(conf_idx, params_path), ...]
for ligand_name, conformers in all_conformers.items():
    all_params[ligand_name] = []
    for i, conf_mol in enumerate(conformers):
        from cdpkit_conformer_pipeline import write_cdpkit_conformer_to_sdf
        sdf_path = os.path.join(conf_params_dir, f"{ligand_name}_{i+1}.sdf")
        write_cdpkit_conformer_to_sdf(conf_mol, sdf_path)
        
        pf = convert_conformer_to_params(
            sdf_path, ligand_name, i+1, conf_params_dir, REALM_LOCATION
        )
        if pf:
            all_params[ligand_name].append((i+1, pf))
    print(f"  {ligand_name}: {len(all_params[ligand_name])} params files")



Centering ligands at (   1.315,    0.213,   -3.003)
Atom names contain duplications -- renaming all atoms.
  Aromatic bonds must be identified explicitly --
  alternating single/double bonds (Kekule structure) won't cut it.
  This warning does not apply to you if your molecule really isn't aromatic.
Total naive charge -2.185, desired charge 0.000, offsetting all atoms by 0.046
Average 47.0 atoms (27.0 non-H atoms) per fragment
(Proteins average 15.5 atoms (7.8 non-H atoms) per residue)
Wrote params file PV-006439055682_1.params


Centering ligands at (   1.315,    0.213,   -3.003)
Atom names contain duplications -- renaming all atoms.
  Aromatic bonds must be identified explicitly --
  alternating single/double bonds (Kekule structure) won't cut it.
  This warning does not apply to you if your molecule really isn't aromatic.
Total naive charge -2.185, desired charge 0.000, offsetting all atoms by 0.046
Average 47.0 atoms (27.0 non-H atoms) per fragment
(Proteins average 15.5 atoms (7.8 non-H atoms) per residue)
Wrote params file PV-006439055682_2.params


Centering ligands at (   1.315,    0.213,   -3.003)
Atom names contain duplications -- renaming all atoms.
  Aromatic bonds must be identified explicitly --
  alternating single/double bonds (Kekule structure) won't cut it.
  This warning does not apply to you if your molecule really isn't aromatic.
Total naive charge -2.185, desired charge 0.000, offsetting all atoms by 0.046
Average 47.0 atoms (27.0 non-H atoms) per fragment
(Proteins average 15.5 atoms (7.8 non-H atoms) per residue)
Wrote params file PV-006439055682_3.params


Centering ligands at (   1.315,    0.213,   -3.003)
Atom names contain duplications -- renaming all atoms.
  Aromatic bonds must be identified explicitly --
  alternating single/double bonds (Kekule structure) won't cut it.
  This warning does not apply to you if your molecule really isn't aromatic.
Total naive charge -2.185, desired charge 0.000, offsetting all atoms by 0.046
Average 47.0 atoms (27.0 non-H atoms) per fragment
(Proteins average 15.5 atoms (7.8 non-H atoms) per residue)
Wrote params file PV-006439055682_4.params


Centering ligands at (   1.315,    0.213,   -3.003)
Atom names contain duplications -- renaming all atoms.
  Aromatic bonds must be identified explicitly --
  alternating single/double bonds (Kekule structure) won't cut it.
  This warning does not apply to you if your molecule really isn't aromatic.
Total naive charge -2.185, desired charge 0.000, offsetting all atoms by 0.046
Average 47.0 atoms (27.0 non-H atoms) per fragment
(Proteins average 15.5 atoms (7.8 non-H atoms) per residue)
Wrote params file PV-006439055682_5.params


...

Centering ligands at (   1.315,    0.213,   -3.003)
Atom names contain duplications -- renaming all atoms.
  Aromatic bonds must be identified explicitly --
  alternating single/double bonds (Kekule structure) won't cut it.
  This warning does not apply to you if your molecule really isn't aromatic.
Total naive charge -2.185, desired charge 0.000, offsetting all atoms by 0.046
Average 47.0 atoms (27.0 non-H atoms) per fragment
(Proteins average 15.5 atoms (7.8 non-H atoms) per residue)
Wrote params file PV-006439055682_146.params


Centering ligands at (   1.315,    0.213,   -3.003)
Atom names contain duplications -- renaming all atoms.
  Aromatic bonds must be identified explicitly --
  alternating single/double bonds (Kekule structure) won't cut it.
  This warning does not apply to you if your molecule really isn't aromatic.
Total naive charge -2.185, desired charge 0.000, offsetting all atoms by 0.046
Average 47.0 atoms (27.0 non-H atoms) per fragment
(Proteins average 15.5 atoms (7.8 non-H atoms) per residue)
Wrote params file PV-006439055682_147.params


Centering ligands at (   1.315,    0.213,   -3.003)
Atom names contain duplications -- renaming all atoms.
  Aromatic bonds must be identified explicitly --
  alternating single/double bonds (Kekule structure) won't cut it.
  This warning does not apply to you if your molecule really isn't aromatic.
Total naive charge -2.185, desired charge 0.000, offsetting all atoms by 0.046
Average 47.0 atoms (27.0 non-H atoms) per fragment
(Proteins average 15.5 atoms (7.8 non-H atoms) per residue)
Wrote params file PV-006439055682_148.params


Centering ligands at (   1.315,    0.213,   -3.003)
Atom names contain duplications -- renaming all atoms.
  Aromatic bonds must be identified explicitly --
  alternating single/double bonds (Kekule structure) won't cut it.
  This warning does not apply to you if your molecule really isn't aromatic.
Total naive charge -2.185, desired charge 0.000, offsetting all atoms by 0.046
Average 47.0 atoms (27.0 non-H atoms) per fragment
(Proteins average 15.5 atoms (7.8 non-H atoms) per residue)
Wrote params file PV-006439055682_149.params


Centering ligands at (   1.315,    0.213,   -3.003)
Atom names contain duplications -- renaming all atoms.
  Aromatic bonds must be identified explicitly --
  alternating single/double bonds (Kekule structure) won't cut it.
  This warning does not apply to you if your molecule really isn't aromatic.
Total naive charge -2.185, desired charge 0.000, offsetting all atoms by 0.046
Average 47.0 atoms (27.0 non-H atoms) per fragment
(Proteins average 15.5 atoms (7.8 non-H atoms) per residue)
Wrote params file PV-006439055682_150.params
  PV-006439055682: 150 params files


In [ ]:
        # ── Copy extra args file to this job's output directory ───────
        args_outfile = os.path.join(work_subdir, "extra_args.txt")
        shutil.copy2(EXTRA_ARG, args_outfile)


Submitting 150 scoring jobs for PV-006439055682...
  PV-006439055682_1: Job <186756> is submitted to queue <long>.
  PV-006439055682_2: Job <186757> is submitted to queue <long>.
  PV-006439055682_3: Job <186758> is submitted to queue <long>.
  PV-006439055682_4: Job <186759> is submitted to queue <long>.
  PV-006439055682_5: Job <186760> is submitted to queue <long>.
  PV-006439055682_6: Job <186761> is submitted to queue <long>.
  PV-006439055682_7: Job <186762> is submitted to queue <long>.
  PV-006439055682_8: Job <186763> is submitted to queue <long>.
  PV-006439055682_9: Job <186764> is submitted to queue <long>.
  PV-006439055682_10: Job <186765> is submitted to queue <long>.
  PV-006439055682_11: Job <186766> is submitted to queue <long>.
  PV-006439055682_12: Job <186767> is submitted to queue <long>.
  PV-006439055682_13: Job <186768> is submitted to queue <long>.
  PV-006439055682_14: Job <186769> is submitted to queue <long>.
  PV-006439055682_15: Job <186770> is submitted

In [ ]:
# %% [markdown]
# ## Step 6b: Collect Rosetta scores after bsub jobs complete

# %%
"""
Run this cell after all bsub jobs have finished.
It parses the Rosetta output PDBs in each work directory and builds the
global_ranked heapq for the Results section below.
"""

import heapq, json

# Load job map
job_file = os.path.join(OUTPUT_DIR, 'bsub_jobs.json')
with open(job_file, "r") as fh:
    job_map = json.load(fh)

# Re-import scoring utilities (available after Step 1 imports)
from cdpkit_conformer_pipeline import parse_placement_scores, compute_weighted_total

global_ranked = []
missing = 0

for conf_name, info in job_map.items():
    ligand_name = info["ligand_name"]
    conf_idx = info["conf_idx"]
    work_subdir = info["work_subdir"]
    
    # Walk the work directory looking for placed PDB files
    scores = {}
    for root, dirs, files in os.walk(work_subdir):
        for f in files:
            if f.endswith(".pdb") and "placed" in f and "minipose" not in f:
                pdb_path = os.path.join(root, f)
                parsed = parse_placement_scores(pdb_path)
                if parsed:
                    total, _ = compute_weighted_total(parsed, {})
                    lig_name = f.replace(".pdb", "")
                    scores[lig_name] = total
    
    if scores:
        best_conf_score = max(scores.values())
        entry = (-best_conf_score, ligand_name, conf_idx, best_conf_score)
        if len(global_ranked) < TOP_N:
            heapq.heappush(global_ranked, entry)
        elif best_conf_score > -global_ranked[0][0]:
            heapq.heapreplace(global_ranked, entry)
        print(f"  {conf_name}: score={best_conf_score:.4f}")
    else:
        missing += 1
        print(f"  {conf_name}: no scores found (job may not be done)")

print(f"\n📊 Collected scores: {len(job_map) - missing}/{len(job_map)} conformers")
print(f"   Missing: {missing} (re-run this cell after jobs complete)")

In [ ]:
# %% [markdown]
# ## Results: Top Ranked Conformers

# %%
sorted_results = sorted(global_ranked, reverse=True)
max_score = max(s[3] for s in sorted_results) if sorted_results else 1.0

print(f"{'Rank':<6} {'Ligand':<30} {'Conf#':<6} {'Score':<12} {'Normalized'}")
print("-" * 70)
for rank, (neg_score, ligand_name, conf_idx, conf_score) in enumerate(sorted_results, 1):
    norm = conf_score / max_score if max_score != 0 else 0
    print(f"{rank:<6} {ligand_name:<30} {conf_idx:<6} {conf_score:<12.4f} {norm:<10.4f}")

# Save to CSV
import csv
output_csv = os.path.join(OUTPUT_DIR, 'ranked_conformers.csv')
with open(output_csv, 'w', newline='') as fh:
    writer = csv.writer(fh)
    writer.writerow(['rank', 'ligand', 'conformer', 'rosetta_score', 'normalized_score'])
    for rank, (neg_score, ligand_name, conf_idx, conf_score) in enumerate(sorted_results, 1):
        norm = conf_score / max_score if max_score != 0 else 0
        writer.writerow([rank, ligand_name, conf_idx, f"{conf_score:.4f}", f"{norm:.4f}"])

print(f"\n✅ Results saved to: {output_csv}")